# EFB: Exclusive Feature Bundling in LightGBM

EFB is a feature engineering optimization that **reduces dimensionality by merging features that rarely take non-zero values simultaneously**. This cuts histogram memory and computation per split without losing predictive power.

## Core Idea

Many datasets have features that are **mutually exclusive**—they don't activate together. EFB bundles these into single features, reducing the feature space dramatically.

### Example

Imagine categorical encoding:
- Feature A: one-hot encoded → `[1, 0, 0, 0]` (one category active)
- Feature B: one-hot encoded → `[0, 1, 0, 0]` (one category active)
- Feature C: one-hot encoded → `[0, 0, 1, 0]` (one category active)

Since exactly one is 1 per sample, they're mutually exclusive. EFB bundles them into a single feature with 12 categories instead of 3 separate binary features.

## Why This Works

### Histogram Construction Cost

For each feature, LightGBM builds a histogram during split-finding:
$$\text{Memory} \propto \text{num\_features} \times \text{max\_bin}$$

With $D$ features and $B$ bins:
- **Without EFB**: $D \times B$ histogram entries per node
- **With EFB**: $D' \times B$ where $D' \ll D$ (bundled features)

Reducing features from 1000 to 200 via bundling = **5× less histogram memory and computation**.

### No Information Loss (Usually)

If features A and B are mutually exclusive:
- The tree split on A is: "if feature A = category X, go left"
- The tree split on B is: "if feature B = category Y, go left"

After bundling into feature AB:
- The split becomes: "if feature AB ∈ {X, Y, ...}, go left"

**The split expressive power is preserved** because the bundled feature carries the same information.

## Algorithm: Detecting Mutual Exclusivity

LightGBM uses a **greedy conflict graph** approach:

1. **Build a conflict graph**: Nodes are features; edges exist if features conflict (co-occur non-zero).
2. **Find independent sets**: Features with no edges between them can bundle together.
3. **Greedy bundling**: Sort features by sparsity; bundle greedily to minimize conflicts.

### Conflict Definition

Two features conflict if they're simultaneously non-zero in the same sample. LightGBM allows a small **conflict rate** (e.g., 10%) to tolerate rare conflicts.

```
Conflict rate = (samples where both A and B are non-zero) / total_samples
```

If conflict rate < threshold (e.g., 0.1), bundle them.

## Bundling Strategy

### 1. **Lossless Bundling** (Zero Conflicts)
Features are truly mutually exclusive → bundle with no information loss.

**Example**: One-hot encoded categoricals
```
Color: [Red=1, Green=0, Blue=0]
Size:  [Small=0, Large=0, Medium=1]
→ Bundle into: ColorSize feature with 9 categories
```

### 2. **Lossy Bundling** (Allowed Conflicts)
Features have rare co-occurrences → bundle anyway, accepting slight loss.

**Example**: Sparse features with 5% conflict rate
```
Feature A: sparse signal (mostly 0)
Feature B: sparse signal (mostly 0, rarely co-occurs with A)
→ Bundle; 5% of samples have both active, but rare
```

The loss is minimal because the tree can usually still separate classes with the bundled feature.

## Memory and Speed Impact

### Example: Credit Card Fraud Dataset

| Scenario | Features | Histogram Size | Training Time |
|----------|----------|---|---|
| **No EFB** | 100 | 100 × 256 bins = 25.6K | 100s |
| **EFB (conflict_rate=0.1)** | 30 bundled | 30 × 256 bins = 7.7K | 20s |
| **Speed-up** | 70% reduction | **70% reduction** | **5× faster** |

For datasets with many categorical features (e.g., PayFac merchant attributes), EFB can deliver **3–10× speedups**.

## Enabling EFB in LightGBM

```python
import lightgbm as lgb
model = lgb.LGBMClassifier(objective='binary',
                           boosting_type='gbdt',
                           metric='auc',
                           learning_rate=0.05,
                           num_leaves=31,
                           max_depth=5,
                            top_rate =  0.2,           # Keep top 20% by gradient
                            other_rate = 0.1,         # Sample 10% of rest

                            # EFB - Exclusive Feature Bundling
                            enable_bundle=True,
                            max_conflict_rate = 0.05, # start small
                            zero_as_missing = False
)
}

train_data = lgb.Dataset(X_train, label=y_train, free_raw_data=False)
model = lgb.train(params, train_data, num_boost_round=100)

# Check bundling in model
print(model.feature_names())  # Will show bundled feature groups
```

### Key Parameters

| Parameter | Default | Effect |
|-----------|---------|--------|
| `enable_bundle` | `True` | Enable/disable EFB |
| `max_conflict_rate` | `0.1` | Max allowed conflict rate (0.0–1.0) |
| `zero_as_missing` | `False` | Treat 0-valued features as missing (helps detect exclusivity) |

**Tip**: Set `zero_as_missing=True` for sparse features (one-hot, embeddings, indicators) to improve bundling.

## In Your SentinelPay Context

### Opportunity: PayFac Merchant Features

Your PayFac sub-merchant portfolio likely has:
- **One-hot categoricals**: Industry, country, payment method, merchant tier
- **Indicator/flag features**: is_high_risk, is_new_merchant, passed_kyc, etc.
- **Sparse signals**: Behavioral flags, rare transaction types

**EFB benefits**:
1. **Industry + Country + Payment Method**: These are often mutually exclusive per merchant → bundle into 50–200 features instead of 500+
2. **Flags**: Binary indicators can be bundled if they don't co-activate
3. **Speed**: Training time from 2 min → 30 sec on large datasets

### When to Use EFB with GOSS

Combine GOSS + EFB for maximum acceleration:
```python
params = {
    # GOSS: Focus on hard cases
    'top_rate': 0.2,
    'other_rate': 0.1,
    
    # EFB: Reduce feature dimensionality
    'enable_bundle': True,
    'max_conflict_rate': 0.1,
    'zero_as_missing': True,
    
    # Result: ~10–15× speedup on large datasets
}
```

## Edge Cases & Gotchas

### 1. **Over-bundling Hurts Accuracy**
If `max_conflict_rate` is too high (e.g., 0.5), you're bundling features that co-occur frequently → information loss.

**Solution**: Start with 0.05–0.1; tune like a hyperparameter.

### 2. **Categorical Encoding Matters**
EFB works best with **sparse, categorical features**. Dense numerical features don't bundle well.

### 3. **Log Inspection**
Enable `verbose=1` to see bundling decisions:
```
[LightGBM: feature_names=[bundled_0, bundled_1, ...]]
```

If bundling is too aggressive, loosen `max_conflict_rate`.

## Key Insight

**EFB is a dimensionality reduction trick tailored to tree-based models.** Unlike PCA or feature selection, EFB preserves tree expressiveness by bundling **related categorical features** that don't conflict. This is why it's so effective for datasets with many one-hot encoded or indicator features—common in fintech and PayFac applications.

Combined with GOSS (instance sampling) and leaf-wise growth, EFB makes LightGBM **3–10× faster than XGBoost** on real-world datasets without sacrificing accuracy.

In [5]:
import lightgbm as lgb
model = lgb.LGBMClassifier(objective='binary',
                           boosting_type='gbdt',
                           metric='auc',
                           learning_rate=0.05,
                           num_leaves=31,
                           max_depth=5,
                            top_rate =  0.2,           # Keep top 20% by gradient
                            other_rate = 0.1,         # Sample 10% of rest

                            # EFB - Exclusive Feature Bundling
                            enable_bundle=True,
                            max_conflict_rate = 0.05, # start small
                            zero_as_missing = False
)